# Curated MIMIC-III Analysis (replica of `mimic-iii-analysis.html`)

This notebook mirrors the workflow and plot types from `mimic-iii-analysis.html`, but computes directly from curated experiment outputs in `data/curated/`.


In [ ]:
import warnings
from pathlib import Path

from IPython.display import display

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_style('whitegrid')


# Setup

In [ ]:
# Runtime configuration
CURATED_ROOT = Path('..') / 'data' / 'curated'
DATAFILE_NAME = 'all_hourly_data.h5'

# Variants (excluding pop5000 by default; it uses a different filename and is much larger)
SCENARIOS = [
    'age35_range5','age35_range10','age35_range15',
    'age45_range5','age45_range10','age45_range15',
    'age55_range5','age55_range10','age55_range15',
    'min48hr','min18age','minperc5',
]

AGG_METHODS = [
    'mean','median','standard deviation','mean deviation','maximum deviation'
]

LABELS = ['icu','mortality']
SEEDS = [22,985,439,81]
TEST_SIZE = 0.2

# Per-scenario patient cap.  Must be large enough that variant patients overlap
# meaningfully with the baseline test split (baseline_nofilters has ~34 k patients).
MAX_RECORDS_PER_SCENARIO = 5000

MODEL_KWARGS = dict(max_iter=500, solver='liblinear', random_state=0)
FALLBACK_BASELINE_SCENARIO = 'baseline_nofilters'

print('Curated root:', CURATED_ROOT.resolve())
print('Variants:', len(SCENARIOS))


In [ ]:
def _get_time_level_name(x_index):
    names = list(x_index.names)
    for candidate in ['hours_in','time','hour','Hour']:
        if candidate in names:
            return candidate
    return names[-1]

def _record_id_levels(x_index):
    t = _get_time_level_name(x_index)
    return [n for n in x_index.names if n != t]

def _select_pos_label(patients_df, target_label):
    if target_label == 'icu':
        return patients_df['mort_icu'].astype(int)
    if target_label == 'mortality':
        if 'mort_hosp' in patients_df.columns:
            return patients_df['mort_hosp'].astype(int)
        if 'hospital_expire_flag' in patients_df.columns:
            return patients_df['hospital_expire_flag'].astype(int)
        raise KeyError('No mortality column found in patients_df')
    raise ValueError('Unknown target label')


# Variant loader (task: `variant-loader`)

In [ ]:
def _pick_datafile(scenario_dir):
    candidates = ['all_hourly_data.h5', 'all_hourly_data_5000.h5', 'all_hourly_data_2000.h5']
    for c in candidates:
        fp = scenario_dir / c
        if fp.exists():
            return fp
    raise FileNotFoundError(f'No all_hourly_data*.h5 found in {scenario_dir}')

def load_curated_scenario(scenario_name, data_root=CURATED_ROOT, load_X=True):
    scenario_dir = data_root / scenario_name
    datafile = _pick_datafile(scenario_dir)

    X = None
    if load_X:
        # Note: vitals_labs is stored in fixed format (can't efficiently filter rows on read).
        # We therefore rely on subsetting rows *after* read and on downsampling the number of records.
        X = pd.read_hdf(str(datafile), 'vitals_labs')

    patients = pd.read_hdf(str(datafile), 'patients')

    try:
        interventions = pd.read_hdf(str(datafile), 'interventions')
    except Exception:
        interventions = None

    return {
        'name': scenario_name,
        'X': X,
        'patients': patients,
        'interventions': interventions,
        'datafile': datafile,
    }

def maybe_sample_record_ids(patients_df, max_records):
    if max_records is None:
        return patients_df
    if len(patients_df) <= max_records:
        return patients_df
    return patients_df.sample(max_records, random_state=0)

def subset_X_by_patients(X, patients_df):
    """Return only time-series rows whose (subject_id,hadm_id,icustay_id) are in patients_df.

    This reduces groupby workload even though `vitals_labs` itself is read in fixed format.

    Assumes X index contains `hours_in` (or one time level) plus record id levels.
    """
    time_level = _get_time_level_name(X.index)
    rec_index = X.index.droplevel(time_level)
    keep = rec_index.isin(patients_df.index)
    return X[keep]

VARIANT_OBJS = []
for s in SCENARIOS:
    print('Loading patients only:', s)
    obj = load_curated_scenario(s, load_X=False)
    obj['patients'] = maybe_sample_record_ids(obj['patients'], MAX_RECORDS_PER_SCENARIO)
    VARIANT_OBJS.append(obj)
print('Loaded variants:', [v['name'] for v in VARIANT_OBJS])


# Baseline loader (task: `duckdb-baseline-export`)

In [ ]:
# Baseline: load from a curated scenario directly (skips DuckDB export)
BASELINE = load_curated_scenario(FALLBACK_BASELINE_SCENARIO, load_X=True)
print('Baseline:', BASELINE['name'], 'X:', BASELINE['X'].shape, 'patients:', BASELINE['patients'].shape)


# Lens plots (task: `lens-plots`)

In [ ]:
LENS_FEATURES = [
    'Systolic blood pressure',
    'Mean blood pressure',
    'Oxygen saturation',
    'Glucose',
    'Hemoglobin',
    'Lactate',
    'pH',
]

def _find_feature_name(feature_map, target_name):
    # Prefer exact case-insensitive match first.
    key = target_name.lower()
    if key in feature_map:
        return feature_map[key]

    # Fallback aliases for common naming variations.
    aliases = {
        'Heart Rate': ['heart rate', 'heartrate', 'pulse'],
        'Systolic blood pressure': ['systolic blood pressure', 'sbp'],
        'Diastolic blood pressure': ['diastolic blood pressure', 'dbp'],
        'Mean blood pressure': ['mean blood pressure', 'map'],
        'Respiratory rate': ['respiratory rate', 'resp rate', 'rr'],
        'Temperature': ['temperature', 'temp'],
        'Oxygen saturation': ['oxygen saturation', 'spo2', 'o2 saturation'],
    }.get(target_name, [target_name.lower()])

    for alias in aliases:
        if alias in feature_map:
            return feature_map[alias]
    return None

def _aggregate_series_by_hour(series, hour_values, method):
    s = pd.Series(series.values, index=hour_values)
    if method == 'mean':
        return s.groupby(level=0).mean()
    if method == 'median':
        return s.groupby(level=0).median()
    if method == 'standard deviation':
        return s.groupby(level=0).std(ddof=1)
    if method == 'mean deviation':
        return s.groupby(level=0).apply(lambda x: (x - x.mean()).abs().mean())
    if method == 'maximum deviation':
        return s.groupby(level=0).apply(lambda x: (x - x.mean()).abs().max())
    raise ValueError(f'Unknown method: {method}')

def build_lens_feature_hour_matrix(X, method, requested_features):
    if not isinstance(X.columns, pd.MultiIndex):
        raise ValueError('Expected MultiIndex columns in vitals_labs data.')

    time_level = _get_time_level_name(X.index)
    hour_values = X.index.get_level_values(time_level)
    all_hours = np.sort(np.asarray(pd.Index(hour_values).unique(), dtype=int))

    # Use only "mean" measurements as the per-hour value for each feature.
    mean_cols = [c for c in X.columns if str(c[1]).lower() == 'mean']
    feature_map = {str(c[0]).lower(): str(c[0]) for c in mean_cols}

    matrix_rows = []
    missing_features = []

    for requested in requested_features:
        actual = _find_feature_name(feature_map, requested)
        if actual is None:
            matrix_rows.append(pd.Series(np.nan, index=all_hours, name=requested))
            missing_features.append(requested)
            continue

        col = next(c for c in mean_cols if str(c[0]) == actual)
        agg = _aggregate_series_by_hour(X[col], hour_values, method)
        agg = agg.reindex(all_hours)
        matrix_rows.append(pd.Series(agg.values, index=all_hours, name=requested))

    matrix = pd.DataFrame(matrix_rows, index=requested_features)
    matrix.columns.name = 'Hour'
    matrix.index.name = 'Feature'
    return matrix, missing_features

def plot_lens_feature_hour_heatmap(matrix, method, missing_features):
    plt.figure(figsize=(14, 6))
    sns.heatmap(matrix, cmap='coolwarm')
    plt.title(f'Lens Plot by Feature and Hour ({method})')
    plt.xlabel('Hour')
    plt.ylabel('Feature')
    if missing_features:
        missing_text = ', '.join(missing_features)
        plt.figtext(0.01, 0.01, f'Missing in dataset: {missing_text}', ha='left', fontsize=9)
    plt.tight_layout()
    plt.show()

# Build lens plots from baseline vitals_labs data.
base_pat = maybe_sample_record_ids(BASELINE['patients'], MAX_RECORDS_PER_SCENARIO)
X_base_small = subset_X_by_patients(BASELINE['X'], base_pat)

for method in AGG_METHODS:
    matrix, missing = build_lens_feature_hour_matrix(X_base_small, method, LENS_FEATURES)
    plot_lens_feature_hour_heatmap(matrix, method, missing)


# Record-level feature aggregation (task: `aggregation-feature-builder`)

In [ ]:
def build_record_aggregated_features(X, method):
    time_level = _get_time_level_name(X.index)
    rec_levels = _record_id_levels(X.index)
    g = X.groupby(level=rec_levels)
    if method == 'mean':
        return g.mean()
    if method == 'median':
        return g.median()
    if method == 'standard deviation':
        return g.std(ddof=1)
    if method == 'mean deviation':
        def _mean_dev(df):
            m = df.mean(axis=0)
            return (df.sub(m, axis=1).abs()).mean(axis=0)
        return g.apply(_mean_dev)
    if method == 'maximum deviation':
        def _max_dev(df):
            m = df.mean(axis=0)
            return (df.sub(m, axis=1).abs()).max(axis=0)
        return g.apply(_max_dev)
    raise ValueError(f'Unknown method: {method}')


# McNemar evaluation + significance (tasks: `mcnemar-eval`, `mcnemar-significance-viz`)

In [ ]:
import gc

def mcnemar_test(pred_base, pred_variant):
    a = np.asarray(pred_base).astype(int)
    b = np.asarray(pred_variant).astype(int)
    n01 = int(np.sum((a == 0) & (b == 1)))
    n10 = int(np.sum((a == 1) & (b == 0)))
    stat = (abs(n01 - n10) - 1.0) ** 2 / (n01 + n10) if (n01 + n10) > 0 else 0.0
    try:
        from scipy.stats import binom
        n = n01 + n10
        if n == 0:
            p = 1.0
        else:
            p_one = binom.cdf(min(n01, n10), n, 0.5)
            p = min(1.0, 2.0 * p_one)
    except Exception:
        p = np.nan
    return stat, p

def impute_and_scale(X):
    X = X.replace([np.inf, -np.inf], np.nan)
    col_means = X.mean(axis=0, skipna=True).fillna(0.0)
    X = X.fillna(col_means).fillna(0.0)
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X.values)
    return Xs, scaler, col_means

# ---------------------------------------------------------------------------
# Pre-compute baseline aggregated features (one-time cost, cached per method)
# ---------------------------------------------------------------------------
_BASE_AGG_CACHE = {}

def _get_base_agg(method):
    """Return (X_base_agg, y_icu, y_mortality) for all baseline patients."""
    if method in _BASE_AGG_CACHE:
        return _BASE_AGG_CACHE[method]
    print(f'  [cache miss] Aggregating baseline features for method={method} ...')
    X_agg = build_record_aggregated_features(BASELINE['X'], method)
    pat = BASELINE['patients']
    y_icu  = _select_pos_label(pat, 'icu')
    y_mort = _select_pos_label(pat, 'mortality')
    _BASE_AGG_CACHE[method] = (X_agg, y_icu, y_mort)
    return _BASE_AGG_CACHE[method]

# ---------------------------------------------------------------------------
# Pre-train a single baseline model on the full population (per method+label)
# ---------------------------------------------------------------------------
_BASE_MODEL_CACHE = {}

def _get_base_model(method, target_label):
    """Train a LogisticRegression on ALL baseline patients (done once)."""
    key = (method, target_label)
    if key in _BASE_MODEL_CACHE:
        return _BASE_MODEL_CACHE[key]
    X_agg, y_icu, y_mort = _get_base_agg(method)
    y = y_icu if target_label == 'icu' else y_mort
    Xs, scaler, col_means = impute_and_scale(X_agg)
    clf = LogisticRegression(**MODEL_KWARGS)
    clf.fit(Xs, y.values)
    _BASE_MODEL_CACHE[key] = (clf, scaler, col_means)
    return _BASE_MODEL_CACHE[key]

# ---------------------------------------------------------------------------
# Main evaluation loop
# ---------------------------------------------------------------------------

def evaluate_for_method(method, target_label):
    X_base_agg, y_base_icu, y_base_mort = _get_base_agg(method)
    y_base_all = y_base_icu if target_label == 'icu' else y_base_mort
    base_pat = BASELINE['patients']

    clf_base, scaler_base, cm_base = _get_base_model(method, target_label)

    # ── Baseline self-evaluation (subsample for speed) ──
    base_sub = maybe_sample_record_ids(base_pat, MAX_RECORDS_PER_SCENARIO)
    y_sub = y_base_all.loc[base_sub.index]
    X_sub = X_base_agg.loc[base_sub.index]

    raw = {k: [] for k in ['train_acc','test_acc','train_f1','test_f1']}
    for seed in SEEDS:
        if len(np.unique(y_sub)) < 2:
            break
        tr, te = train_test_split(
            base_sub.index, test_size=TEST_SIZE,
            random_state=seed, stratify=y_sub.values)
        Xtr, sc, cm = impute_and_scale(X_sub.loc[tr])
        Xte = X_sub.loc[te].replace([np.inf,-np.inf], np.nan).fillna(cm).fillna(0.0)
        clf = LogisticRegression(**MODEL_KWARGS)
        clf.fit(Xtr, y_sub.loc[tr].values)
        p_tr = clf.predict(Xtr)
        p_te = clf.predict(sc.transform(Xte.values))
        raw['train_acc'].append(accuracy_score(y_sub.loc[tr].values, p_tr))
        raw['test_acc'].append(accuracy_score(y_sub.loc[te].values, p_te))
        raw['train_f1'].append(f1_score(y_sub.loc[tr].values, p_tr, zero_division=0))
        raw['test_f1'].append(f1_score(y_sub.loc[te].values, p_te, zero_division=0))

    results = {
        'raw': {
            'train_acc': float(np.mean(raw['train_acc'])) if raw['train_acc'] else np.nan,
            'test_acc':  float(np.mean(raw['test_acc']))  if raw['test_acc']  else np.nan,
            'train_f1':  float(np.mean(raw['train_f1']))  if raw['train_f1']  else np.nan,
            'test_f1':   float(np.mean(raw['test_f1']))   if raw['test_f1']   else np.nan,
            'mcnemar': None,
            'base_test_acc': None,
        }
    }

    # ── Variant evaluation ──
    for v in VARIANT_OBJS:
        name = v['name']
        print(f'  Variant={name}  target={target_label}  method={method}')
        v_pat = v['patients']

        common_ids = v_pat.index.intersection(base_pat.index)
        n_common = len(common_ids)
        print(f'    Patients in variant: {len(v_pat)},  overlap with baseline: {n_common}')

        nan_row = {'train_acc':np.nan,'test_acc':np.nan,
                    'train_f1':np.nan,'test_f1':np.nan,
                    'mcnemar':(np.nan,np.nan),'base_test_acc':np.nan}

        if n_common < 30:
            print('    Skipping (too few common patients)')
            results[name] = nan_row; continue

        common_pat = v_pat.loc[common_ids]
        common_sub = maybe_sample_record_ids(common_pat, MAX_RECORDS_PER_SCENARIO)
        cids = common_sub.index
        y_common = _select_pos_label(common_sub, target_label)

        if len(np.unique(y_common)) < 2:
            print('    Skipping (single class)')
            results[name] = nan_row; continue

        # Load variant features for these patients
        X_v = pd.read_hdf(str(v['datafile']), 'vitals_labs')
        X_v = subset_X_by_patients(X_v, common_sub)
        X_v_agg = build_record_aggregated_features(X_v, method)
        del X_v; gc.collect()

        valid = cids.intersection(X_v_agg.index).intersection(X_base_agg.index)
        X_v_agg = X_v_agg.loc[valid].reindex(columns=X_base_agg.columns, fill_value=0)
        y_valid = y_common.loc[valid]

        if len(valid) < 30 or len(np.unique(y_valid)) < 2:
            results[name] = nan_row; del X_v_agg; gc.collect(); continue

        met = {k: [] for k in ['train_acc','test_acc','train_f1','test_f1']}
        b_accs = []
        mstats = []

        for seed in SEEDS:
            tr, te = train_test_split(
                valid, test_size=TEST_SIZE,
                random_state=seed, stratify=y_valid.values)
            y_tr = y_valid.loc[tr].values
            y_te = y_valid.loc[te].values
            if len(np.unique(y_tr)) < 2 or len(te) < 10:
                continue

            # Baseline model → predict on test patients (using baseline features)
            X_te_b = X_base_agg.loc[te].replace([np.inf,-np.inf], np.nan)
            X_te_b = X_te_b.fillna(cm_base).fillna(0.0)
            pred_base = clf_base.predict(scaler_base.transform(X_te_b.values))
            b_accs.append(accuracy_score(y_te, pred_base))

            # Variant model → train on variant training patients only
            Xtr_v, sc_v, cm_v = impute_and_scale(X_v_agg.loc[tr])
            clf_v = LogisticRegression(**MODEL_KWARGS)
            clf_v.fit(Xtr_v, y_tr)
            p_tr_v = clf_v.predict(Xtr_v)
            Xte_v = X_v_agg.loc[te].replace([np.inf,-np.inf], np.nan)
            Xte_v = Xte_v.fillna(cm_v).fillna(0.0)
            pred_var = clf_v.predict(sc_v.transform(Xte_v.values))

            met['train_acc'].append(accuracy_score(y_tr, p_tr_v))
            met['test_acc'].append(accuracy_score(y_te, pred_var))
            met['train_f1'].append(f1_score(y_tr, p_tr_v, zero_division=0))
            met['test_f1'].append(f1_score(y_te, pred_var, zero_division=0))

            stat, p = mcnemar_test(pred_base, pred_var)
            mstats.append((stat, p))

        results[name] = {
            'train_acc': float(np.mean(met['train_acc'])) if met['train_acc'] else np.nan,
            'test_acc':  float(np.mean(met['test_acc']))  if met['test_acc']  else np.nan,
            'train_f1':  float(np.mean(met['train_f1']))  if met['train_f1']  else np.nan,
            'test_f1':   float(np.mean(met['test_f1']))   if met['test_f1']   else np.nan,
            'mcnemar': (
                float(np.mean([s for s,_ in mstats])),
                float(np.min ([p for _,p in mstats])),
            ) if mstats else (np.nan, np.nan),
            'base_test_acc': float(np.mean(b_accs)) if b_accs else np.nan,
        }
        del X_v_agg; gc.collect()

    return results

# ---------------------------------------------------------------------------
# Display helpers
# ---------------------------------------------------------------------------

def show_results(results_dict, target_label, method):
    names = ['raw'] + [n for n in results_dict if n != 'raw']
    rows = []
    for n in names:
        r = results_dict[n]
        if n == 'raw' or r['mcnemar'] is None:
            m_str = '—'
        else:
            stat, p = r['mcnemar']
            m_str = f'{stat:.2f} (p={p:.3e})' if not np.isnan(stat) else 'n/a'
        ba = r.get('base_test_acc')
        ba_str = f'{ba:.3f}' if ba is not None and not np.isnan(ba) else '—'
        rows.append([n, r['train_acc'], r['test_acc'], r['train_f1'],
                      r['test_f1'], ba_str, m_str])
    df = pd.DataFrame(rows, columns=[
        'Scenario','Var Train Acc','Var Test Acc','Var Train F1',
        'Var Test F1','Base Test Acc','McNemar (stat, p)'])
    print(f'\n{target_label.upper()} — McNemar Results ({method})')
    display(df)
    return df

def visualize_mcnemar_significance(results_dict, target_label, method):
    names = ['raw'] + [n for n in results_dict if n != 'raw']
    pvals = []
    for n in names:
        if n == 'raw' or results_dict[n]['mcnemar'] is None:
            pvals.append(np.nan)
        else:
            _, p = results_dict[n]['mcnemar']
            pvals.append(p)
    vals = [-np.log10(max(float(p), 1e-300)) if p == p else np.nan for p in pvals]
    df = pd.DataFrame([vals], columns=names, index=['-log10(p)'])
    plt.figure(figsize=(14, 2.8))
    sns.heatmap(df, cmap='magma', annot=True, fmt='.1f')
    plt.title(f'{target_label.upper()} McNemar Significance ({method})')
    plt.tight_layout(); plt.show()

# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------
run_methods = AGG_METHODS[:1]
for method in run_methods:
    for target_label in LABELS:
        res = evaluate_for_method(method, target_label)
        show_results(res, target_label=target_label, method=method)
        visualize_mcnemar_significance(res, target_label=target_label, method=method)


# Centroid shift (task: `centroid-shift`)

In [ ]:
from scipy.spatial.distance import cosine as cosine_dist

def compute_centroid(X_agg, patients_df, target_label):
    y = _select_pos_label(patients_df, target_label)
    y = y.loc[X_agg.index]
    sel = y.values == 1
    if sel.sum() == 0:
        return None
    vec = X_agg.loc[y.index[sel]].mean(axis=0).values
    vec = np.nan_to_num(vec, nan=0.0, posinf=0.0, neginf=0.0)
    return vec

def visualize_centroid_shift(method):
    X_base_agg, _, _ = _get_base_agg(method)
    base_pat = BASELINE['patients']

    cents_base = {lab: compute_centroid(X_base_agg, base_pat, lab) for lab in LABELS}

    variant_centroids = []
    for v in VARIANT_OBJS:
        pat = maybe_sample_record_ids(v['patients'], MAX_RECORDS_PER_SCENARIO)
        X_v = pd.read_hdf(str(v['datafile']), 'vitals_labs')
        Xv_small = subset_X_by_patients(X_v, pat)
        Xv_agg = build_record_aggregated_features(Xv_small, method).loc[pat.index]
        Xv_agg = Xv_agg.reindex(columns=X_base_agg.columns, fill_value=0)
        cents = {lab: compute_centroid(Xv_agg, pat, lab) for lab in LABELS}
        variant_centroids.append({'name': v['name'], 'centroids': cents})
        del X_v, Xv_small, Xv_agg; gc.collect()

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, lab in zip(axes, LABELS):
        if cents_base[lab] is None:
            ax.set_title(f'{lab} (no baseline centroid)')
            continue

        vecs, names, dists = [], [], []
        vecs.append(cents_base[lab])
        names.append('baseline')
        dists.append(0.0)

        for vc in variant_centroids:
            vec = vc['centroids'].get(lab)
            if vec is None:
                continue
            vecs.append(vec)
            names.append(vc['name'])
            euc = float(np.linalg.norm(vec - cents_base[lab]))
            cos = float(cosine_dist(cents_base[lab], vec))
            dists.append(euc)
            print(f'  {lab} | {vc["name"]:20s}  euclid={euc:.4f}  cosine={cos:.4f}')

        cleaned = [np.nan_to_num(v, nan=0.0) for v in vecs]
        if len(cleaned) < 2:
            ax.set_title(f'{lab} (insufficient variants)')
            continue

        stacked = np.vstack(cleaned)
        scaler = StandardScaler()
        stacked_s = scaler.fit_transform(stacked)
        Z = PCA(n_components=2, random_state=0).fit_transform(stacked_s)

        base2 = Z[0]
        ax.scatter(Z[1:, 0], Z[1:, 1], s=50, zorder=3)
        ax.scatter([base2[0]], [base2[1]], marker='*', s=200, c='red',
                   zorder=4, label='baseline')
        for i in range(1, len(Z)):
            ax.annotate(names[i], (Z[i, 0], Z[i, 1]), fontsize=7, alpha=0.8)
            ax.plot([base2[0], Z[i, 0]], [base2[1], Z[i, 1]],
                    linewidth=0.8, alpha=0.5, color='grey')
        ax.set_title(f'{lab} centroid shift')
        ax.set_xlabel('PC1')
        ax.set_ylabel('PC2')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.suptitle(f'Centroid Shift (aggregation={method})', fontsize=13)
    plt.tight_layout()
    plt.show()

    # Summary table
    rows = []
    for lab in LABELS:
        if cents_base[lab] is None:
            continue
        for vc in variant_centroids:
            vec = vc['centroids'].get(lab)
            if vec is None:
                rows.append([lab, vc['name'], np.nan, np.nan])
                continue
            euc = float(np.linalg.norm(vec - cents_base[lab]))
            cos = float(cosine_dist(cents_base[lab], vec))
            rows.append([lab, vc['name'], euc, cos])
    df = pd.DataFrame(rows, columns=['Label','Scenario','Euclidean','Cosine'])
    display(df)

for method in AGG_METHODS[:1]:
    visualize_centroid_shift(method)


# Extra visuals (task: `extra-visuals`)

In [ ]:
def fenceposts_to_record_lengths(scenario_dir, max_records=None):
    fp_path = scenario_dir / 'fenceposts.npy'
    fp = np.load(str(fp_path))
    if max_records is not None and len(fp) > max_records:
        rng = np.random.default_rng(0)
        fp = fp[rng.choice(len(fp), size=max_records, replace=False)]
    return fp

rows = []
for v in [BASELINE] + VARIANT_OBJS:
    # Prevalence
    patients = v['patients']
    for lab in LABELS:
        y = _select_pos_label(patients, lab)
        rows.append({'scenario': v['name'], 'target': lab, 'prevalence': float(y.mean())})

df_prev = pd.DataFrame(rows)
plt.figure(figsize=(14, 6))
sns.barplot(data=df_prev, x='scenario', y='prevalence', hue='target')
plt.title('Positive-class prevalence by scenario')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 7))
for v in [BASELINE] + VARIANT_OBJS:
    scenario_dir = Path(v['datafile']).parent
    lens = fenceposts_to_record_lengths(scenario_dir, max_records=MAX_RECORDS_PER_SCENARIO)
    sns.kdeplot(lens, label=v['name'], warn_singular=False, clip=(0, None))
plt.title('Distribution of record lengths')
plt.xlabel('record_length (max hours_in + 1)')
plt.legend(fontsize=8)
plt.show()
